<a href="https://colab.research.google.com/github/Sveshchaudhary/quantium-data-analytics-simulation/blob/main/task2_trial_store_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
Uploaded = files.upload()

In [ ]:
import pandas as pd
df = pd.read_csv('QVI_data.csv')

In [ ]:
df.head()

In [ ]:
df.info

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.describe()

In [ ]:
df.columns.tolist()

In [ ]:
df['DATE'] = pd.to_datetime(df['DATE'])
df['YEARMONTH'] = df['DATE'].dt.to_period('M')

In [ ]:
monthly_metrics = df.groupby(['STORE_NBR', 'YEARMONTH']).agg(
    total_sales=('TOT_SALES', 'sum'),
    total_customers=('LYLTY_CARD_NBR', 'nunique'),
    total_transactions=('TXN_ID', 'nunique')
).reset_index()

monthly_metrics['avg_txn_per_customer'] = monthly_metrics['total_transactions'] / monthly_metrics['total_customers']

monthly_metrics.head()

In [ ]:
monthly_metrics['YEARMONTH'].unique()

In [ ]:
pre_trial = monthly_metrics[monthly_metrics['YEARMONTH'] < '2019-02']
trial_period = monthly_metrics[(monthly_metrics['YEARMONTH'] >= '2019-02') & (monthly_metrics['YEARMONTH'] <= '2019-04')]

pre_trial.shape, trial_period.shape

In [ ]:
pre_trial_sales = pre_trial.pivot(index='STORE_NBR', columns='YEARMONTH', values='total_sales')
pre_trial_sales.head()

In [ ]:
import numpy as np

def calculate_control_scores(trial_store, pre_trial_sales):
    trial_data = pre_trial_sales.loc[trial_store]
    scores = []

    for store in pre_trial_sales.index:
        if store == trial_store:
            continue
        store_data = pre_trial_sales.loc[store]

        # Correlation: how similarly the two stores' sales move over time
        corr = np.corrcoef(trial_data, store_data)[0, 1]

        # Magnitude distance: how close the actual sales values are
        distance = np.abs(trial_data - store_data).sum()
        max_dist = np.abs(trial_data.max() - store_data.min())
        min_dist = 0
        magnitude_score = 1 - (distance - min_dist) / (max_dist - min_dist + 1e-9)

        # Combine both into one overall score (weighted average)
        overall_score = (corr * 0.5) + (magnitude_score * 0.5)

        scores.append({'STORE_NBR': store, 'corr_score': corr, 'magnitude_score': magnitude_score, 'overall_score': overall_score})

    return pd.DataFrame(scores).sort_values('overall_score', ascending=False)

In [ ]:
control_scores_77 = calculate_control_scores(77, pre_trial_sales)
control_scores_77.head()

In [ ]:
def calculate_control_scores(trial_store, pre_trial_sales):
    trial_data = pre_trial_sales.loc[trial_store]
    scores = []

    for store in pre_trial_sales.index:
        if store == trial_store:
            continue
        store_data = pre_trial_sales.loc[store]

        # Correlation
        corr = np.corrcoef(trial_data, store_data)[0, 1]

        # Magnitude distance (corrected)
        distance = np.abs(trial_data - store_data).sum()
        scores.append({'STORE_NBR': store, 'corr_score': corr, 'distance': distance})

    result = pd.DataFrame(scores)
    # Normalize distance to a 0-1 similarity score, where 1 = most similar
    result['magnitude_score'] = 1 - (result['distance'] - result['distance'].min()) / (result['distance'].max() - result['distance'].min())
    result['overall_score'] = (result['corr_score'] * 0.5) + (result['magnitude_score'] * 0.5)

    return result.sort_values('overall_score', ascending=False)

In [ ]:
control_scores_77 = calculate_control_scores(77, pre_trial_sales)
control_scores_77.head()

In [ ]:
control_scores_86 = calculate_control_scores(86, pre_trial_sales)
control_scores_86.head()

In [ ]:
control_scores_88 = calculate_control_scores(88, pre_trial_sales)
control_scores_88.head()

In [ ]:
def compare_trial_control(trial_store, control_store, monthly_metrics):
    trial_data = monthly_metrics[monthly_metrics['STORE_NBR'] == trial_store]
    control_data = monthly_metrics[monthly_metrics['STORE_NBR'] == control_store]

    comparison = trial_data.merge(control_data, on='YEARMONTH', suffixes=('_trial', '_control'))
    comparison['sales_diff_pct'] = ((comparison['total_sales_trial'] - comparison['total_sales_control']) / comparison['total_sales_control']) * 100

    return comparison[['YEARMONTH', 'total_sales_trial', 'total_sales_control', 'sales_diff_pct']]

In [ ]:
compare_77 = compare_trial_control(77, 233, monthly_metrics)
compare_77

In [ ]:
compare_86 = compare_trial_control(86, 155, monthly_metrics)
compare_86

In [ ]:
compare_88 = compare_trial_control(88, 125, monthly_metrics)
compare_88

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(10, 12))

for ax, (trial, control, data) in zip(axes, [(77, 233, compare_77), (86, 155, compare_86), (88, 125, compare_88)]):
    ax.plot(data['YEARMONTH'].astype(str), data['total_sales_trial'], label=f'Trial Store {trial}', marker='o')
    ax.plot(data['YEARMONTH'].astype(str), data['total_sales_control'], label=f'Control Store {control}', marker='o')
    ax.axvspan('2019-02', '2019-04', alpha=0.2, color='yellow', label='Trial Period')
    ax.set_title(f'Store {trial} vs Control {control}')
    ax.legend()
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
monthly_metrics[(monthly_metrics['STORE_NBR'].isin([77, 233])) & (monthly_metrics['YEARMONTH'] >= '2019-02') & (monthly_metrics['YEARMONTH'] <= '2019-04')]

In [ ]:
store_77 = monthly_metrics[monthly_metrics['STORE_NBR'] == 77]
store_233 = monthly_metrics[monthly_metrics['STORE_NBR'] == 233]

In [ ]:
ax.plot(data['YEARMONTH'].astype(str), data['total_sales_trial'], label=..., marker='o')
ax.plot(data['YEARMONTH'].astype(str), data['total_sales_control'], label=..., marker='o')

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(store_77['YEARMONTH'].astype(str), store_77['total_customers'], label='Store 77 (Trial)', marker='o')
plt.plot(store_233['YEARMONTH'].astype(str), store_233['total_customers'], label='Store 233 (Control)', marker='o')
plt.title('Customer Count: Store 77 vs Control 233')
plt.xlabel('Month')
plt.ylabel('Number of Customers')
plt.legend()
plt.xticks(rotation=45)
plt.show()

In [ ]:
df.to_csv('QVI_data.csv', index=False)
from google.colab import files
files.download('QVI_data.csv')

## Summary of Findings — Trial Store Evaluation

This analysis evaluated the performance of a store trial conducted in stores 77, 86,
and 88, using matched control stores identified through a combination of Pearson
correlation and magnitude-distance scoring on pre-trial (Jul 2018-Jan 2019) monthly
sales data.

### Control Store Matches
- Store 77 → Control Store 233 (similarity score: 0.95)
- Store 86 → Control Store 155 (similarity score: 0.94)
- Store 88 → Control Store 125 (similarity score: 0.76 — weaker match)

### Trial Period Results (Feb-Apr 2019)
- **Store 77**: Strong, growing positive effect (+40% to +66% vs control by March-April),
  with trial and control tracking closely pre-trial. Clear evidence the trial drove
  incremental sales.
- **Store 86**: Weak, inconsistent effect — only March showed a notable spike (+28%);
  February and April were roughly flat vs control. Limited evidence of a sustained
  trial impact.
- **Store 88**: Large, consistent gap vs control (+20% to +50%) throughout the trial.
  However, store 88 was already outperforming its control before the trial began,
  and control match quality was weaker — this result should be interpreted with
  caution, as some of the gap may reflect pre-existing store differences rather
  than the trial itself.

### Driver Analysis (Store 77 Example)
For Store 77, the sales increase was primarily driven by stronger customer retention
rather than increased purchase frequency. While average transactions per customer
remained similar between Store 77 and its control (both near 1.0), Store 77's
customer count held steady during the trial period, while Control Store 233's
customer count declined sharply (from 45 to 30 customers). This suggests the trial
may have helped retain customers who would otherwise have been lost, rather than
attracting a large influx of new shoppers.
### Recommendation
The trial shows strong, credible evidence of success in Store 77. Store 88's result
is promising but less reliable due to a weaker control match and pre-existing
divergence. Store 86 shows minimal evidence of a meaningful trial effect.
Recommend a broader rollout be piloted cautiously, prioritizing conditions similar
to Store 77, while further investigating why Store 86 did not respond and
sourcing a better control match for Store 88 before drawing firm conclusions there.

## Note on AI Assistance
This project was completed with AI assistance (Claude) for code guidance, debugging
support, and explanations of statistical concepts (e.g., Pearson correlation, control
store matching). All code outputs, control store selections, and trial-vs-control
comparisons were reviewed and interpreted independently. Statistical caveats (e.g.,
weaker control match for Store 88, pre-existing divergence before the trial period)
were identified and incorporated into the final recommendation based on critical
review of the results.